# Hand Detection - MediaPipe Label Generation (CPU Notebook)

This notebook **only** runs MediaPipe to generate bounding-box labels and saves them to Google Drive.
Run this first on a **CPU** runtime (no GPU/TPU needed).

Steps:
1. Mounts Google Drive
2. Installs MediaPipe
3. Imports & global constants
4. Collects image paths from the dataset folder
5. Runs MediaPipe Hands over the dataset
6. Saves the label cache (`labels_cache_data.npz`) to `modelForHands/` on Drive

After this notebook finishes, open **`hand_detection_cnn_train.ipynb`** on a **GPU** runtime to train the model.

> **Runtime -> Change runtime type -> CPU** (default) before running.

## Cell 1 - Mount Google Drive

In [ ]:
from google.colab import drive
# Standard Colab mount point: Drive root appears at /content/drive
# Your files will then be accessible at /content/drive/MyDrive/...
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

## Cell 2 - Install / upgrade dependencies

In [ ]:
# MediaPipe 0.10.x ships with its own bundled protobuf; keep it isolated.
!pip install -q mediapipe opencv-python-headless
print('Dependencies ready.')

## Cell 3 - Imports & global constants

In [ ]:
import os
import random
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path

# -- Paths -----------------------------------------------------------------------
# Standard Colab Google Drive mount: /content/drive/MyDrive/<your folder>
DATASET_ROOT   = '/content/drive/MyDrive/ML-CEP/data'
CHECKPOINT_DIR = '/content/drive/MyDrive/ML-CEP/modelForHands'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -- Reproducibility seed --------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'MediaPipe {mp.__version__}')
print(f'DATASET_ROOT   : {DATASET_ROOT}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')

## Cell 4 - Collect image paths from the dataset folder

Expected layout:
```
data/
  1/
    manahil/  sitwat/  Marium/  talha/
  2/ ...
  26/ ...
```

In [ ]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def collect_image_paths(root: str) -> list:
    """Walk dataset root and return sorted list of image file paths."""
    paths = []
    root_path = Path(root)
    for main_folder in sorted(root_path.iterdir()):
        if not main_folder.is_dir():
            continue
        for sub_folder in sorted(main_folder.iterdir()):
            if not sub_folder.is_dir():
                continue
            for img_file in sorted(sub_folder.iterdir()):
                if img_file.suffix.lower() in VALID_EXTENSIONS:
                    paths.append(str(img_file))
    return paths


all_paths = collect_image_paths(DATASET_ROOT)
print(f'Total image files found: {len(all_paths)}')
if all_paths:
    print('Example:', all_paths[0])

## Cell 5 - MediaPipe Hands label generator (streaming / low-RAM)

Processes one image at a time - never loads the full dataset into RAM.
Yields `(image_path, [x, y, w, h])` tuples where all values are **normalised to [0, 1]**.

In [ ]:
def get_hand_bbox_normalised(image_bgr: np.ndarray, mp_hands_module):
    """
    Run MediaPipe Hands on a BGR image.
    Returns normalised (x, y, w, h) of the bounding box, or None if no hand found.
    x, y -> top-left corner  |  w, h -> width / height  (all in [0, 1])
    """
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = mp_hands_module.process(image_rgb)

    if not results.multi_hand_landmarks:
        return None

    # Use the first detected hand
    landmarks = results.multi_hand_landmarks[0]

    xs = [lm.x for lm in landmarks.landmark]
    ys = [lm.y for lm in landmarks.landmark]

    # Clamp to [0, 1] in case MediaPipe goes slightly out of bounds
    x_min = max(0.0, min(xs))
    y_min = max(0.0, min(ys))
    x_max = min(1.0, max(xs))
    y_max = min(1.0, max(ys))

    # Add padding (10% of normalised image size) around the tight landmark box
    pad = 0.1
    x_min = max(0.0, x_min - pad)
    y_min = max(0.0, y_min - pad)
    x_max = min(1.0, x_max + pad)
    y_max = min(1.0, y_max + pad)

    w = x_max - x_min
    h = y_max - y_min

    if w <= 0 or h <= 0:
        return None

    return [x_min, y_min, w, h]


def generate_labels(image_paths: list, verbose_every: int = 500):
    """
    Generator - yields (path_str, bbox_list) for every image where
    MediaPipe detects a hand.  Images with no detection are skipped.
    """
    mp_hands_module = mp.solutions.hands
    with mp_hands_module.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=0.3) as hands:

        skipped = 0
        for idx, path in enumerate(image_paths):
            if verbose_every and idx % verbose_every == 0:
                print(f'  Processed {idx}/{len(image_paths)}, skipped so far: {skipped}')

            img = cv2.imread(path)
            if img is None:
                skipped += 1
                continue

            bbox = get_hand_bbox_normalised(img, hands)
            if bbox is None:
                skipped += 1
                continue

            yield path, bbox

        print(f'Done. Total images: {len(image_paths)}, skipped (no hand): {skipped}')


print('Label generator defined.')

## Cell 6 - Run MediaPipe over the full dataset and save a label file

Results are cached to a `.npz` file on Drive so you don't have to re-run MediaPipe after a session restart.

In [ ]:
LABEL_CACHE = os.path.join(CHECKPOINT_DIR, 'labels_cache_data.npz')

if os.path.exists(LABEL_CACHE):
    print('Loading labels from cache ...')
    cache = np.load(LABEL_CACHE, allow_pickle=True)
    labelled_paths = list(cache['paths'])
    labelled_boxes = list(cache['boxes'])
    print(f'  Loaded {len(labelled_paths)} labelled samples from cache.')
else:
    print('Running MediaPipe Hands on dataset (this may take ~5-15 min) ...')
    labelled_paths = []
    labelled_boxes = []

    for p, bbox in generate_labels(all_paths):
        labelled_paths.append(p)
        labelled_boxes.append(bbox)

    # Save cache
    np.savez_compressed(
        LABEL_CACHE,
        paths=np.array(labelled_paths),
        boxes=np.array(labelled_boxes, dtype=np.float32)
    )
    print(f'Cache saved to {LABEL_CACHE}')

labelled_paths = np.array(labelled_paths)
labelled_boxes = np.array(labelled_boxes, dtype=np.float32)

print(f'\nLabelled dataset size : {len(labelled_paths)}')
print(f'Boxes array shape     : {labelled_boxes.shape}')
print(f'\nLabel cache saved to  : {LABEL_CACHE}')
print('\nDone! Now open hand_detection_cnn_train.ipynb on a GPU runtime to train the model.')